In [ ]:
import cv2
import json
import glob
import os
import numpy as  np
import pandas as pd
import pytesseract
import easyocr
import matplotlib.pyplot as plt

In [ ]:
files = glob.glob("../data/raw/*.png", recursive=True)
print(len(files))

In [ ]:
file_to_remove = "../data/raw/sample.png" # Note: "sample.png", files = [f for f in files if os.path.basename(f) != file_to_remove]
files = [f for f in files if f != file_to_remove]
print(len(files))

In [ ]:
files = sorted(files)

In [ ]:
reader = easyocr.Reader(['en'])
labels = {}
for f in files:
    img = cv2.imread(f)
    result = reader.readtext(img, detail=0)
    text = " ".join(result).lower()
    if "VICTORY" in text:
        labels[f] = 1
    elif "DEFEAT" in text:
        labels[f] = 0
    else:
        labels[f] = None

In [ ]:
# Note: Doesn't work... need to manually label
n = 5
for i, (k, v) in enumerate(labels.items()):
    if i >= n:
        break
    print(k, v)

In [ ]:
df = pd.DataFrame({
    "file": files,
    "basename": [os.path.basename(f) for f in files],
    "label": "" # manually fill 1 = victory, 0 = defeat
})

In [ ]:
df.to_csv("../data/raw/mystery_heroes.csv", index=False)

In [ ]:
# Important: Manually labelled the target column 'label', Victory=1, Defeat=0, on GitHub

In [ ]:
df_labeled = pd.read_csv("../data/raw/labels.csv")

In [ ]:
df_labeled.head()

In [ ]:
df_cleaned = df_labeled.copy()
df_cleaned = df_cleaned[df_cleaned["label"].notna()]
df_cleaned = df_cleaned[df_cleaned["label"] != ""]
df_cleaned = df_cleaned[df_cleaned["label"] != "None"]

In [ ]:
df_cleaned["label"] = df_cleaned["label"].astype(int)
df_cleaned["label"].dtype

In [ ]:
df_cleaned.to_csv("../data/processed/data.csv", index=False)

In [ ]:
df_cleaned.head()

In [ ]:
def show(im, figsize=(4, 2)):
    plt.figure(figsize=figsize)
    cmap = "gray" if im.ndim == 2 else None
    disp = im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    plt.imshow(disp, cmap=cmap)
    plt.axis("off")
    plt.show()

In [ ]:
def preprocess(cell: np.ndarray, scale: int = 3) -> np.ndarray:
    big = cv2.resize(cell, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(big, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(thr) < 127:
        thr = cv2.bitwise_not(thr)
    return thr

def read_number(cell: np.ndarray) -> str:
    c = "--psm 7 -c tessedit_char_whitelist=0123456789,"
    return pytesseract.image_to_string(preprocess(cell), config=c).strip().replace(',', '')

def read_name(cell: np.ndarray) -> str:
    reader = easyocr.Reader(["en"], gpu=True)
    res = reader.readtext(cell, detail=0)
    return res
    
def extract_team(img, cfg, team):
    h = cfg["row_h"]
    nx1, nx2 = cfg["name_x"]
    rows = []
    for top in cfg["teams"][team]["row_tops"]:
        rec = {"team": team,
               "name": read_name(img[top:top+h, nx1:nx2])}
        for col, (x1, x2) in cfg["col_x"].items():
            rec[col] = read_number(img[top:top+h, x1:x2])
        rows.append(rec)
    return rows

In [ ]:
cfg = json.load(open("../configs/template.json"))

In [ ]:
records = []
for row in df_cleaned.itertuples(index=False):
    img = cv2.imread(row.file)
    records.append(extract_team(img, cfg, "blue") + extract_team(img, cfg, "red"))
    

In [ ]:
# df_blue = pd.DataFrame(records)
df_extracted = pd.DataFrame(records)

In [ ]:
# df_blue.to_csv("../data/processed/df_blue.csv")
df_extracted.to_csv("../data/processed/extracted_data.csv")

In [ ]:
# TODO: sqlite3 -> conn; df.to_sql("labels", conn, if_exists='replace', index=True)